# Feature engineering
All features come from `src.features.build.build_features(tx, cutoff, cfg)` using only data before `cutoff`.

In [1]:
import sys, warnings, json
sys.path.insert(0, '..'); warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.utils import load_config, load
from src.features.build import build_features, build_labels, data_end, feature_columns
cfg = load_config(); FIG = '../docs/figures/'
import os; os.makedirs(FIG, exist_ok=True)
sns.set_theme(style='whitegrid', palette=['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6'])
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
tx = pd.read_parquet(cfg['paths']['interim'])

In [2]:
cutoff = data_end(tx) - pd.Timedelta(days=90)
f = build_features(tx, cutoff, cfg)
print(cutoff.date(), f.shape)
f.describe().T.round(2)

2011-09-11 (5256, 39)


,count,mean,min,25%,50%,75%,max,std
recency_days,5256.0,206.92656,1.0,50.0,164.0,325.0,648.0,174.401626
tenure_days,5256.0,431.671233,1.0,311.0,474.0,587.0,648.0,180.69029
frequency,5256.0,5.711187,1.0,1.0,3.0,6.0,284.0,11.214046
avg_order_value,5256.0,364.85255,1.55,173.84625,275.021875,406.989167,14844.766667,532.712585
avg_basket_items,5256.0,244.313492,1.0,88.0,149.0,254.0,87167.0,1402.220678
avg_basket_lines,5256.0,20.902822,1.0,9.714286,17.0,26.8,220.0,17.555705
gap_mean,3681.0,90.117896,0.0,38.0,67.125,116.4,632.0,81.248828
gap_std,2791.0,58.451582,0.0,25.13726,45.817027,78.332716,440.527525,47.781736
orders_last_90d,5256.0,0.74277,0.0,0.0,0.0,1.0,45.0,1.785139
revenue_last_90d,5256.0,345.752161,0.0,0.0,0.0,261.28,62151.7,1944.032088


## Leakage check
Rebuilding the table with every post-cutoff row removed must give identical features (also enforced in `tests/test_features.py`).

In [3]:
pd.testing.assert_frame_equal(f, build_features(tx[tx.invoice_date < cutoff], cutoff, cfg)); 'no leakage'

'no leakage'

## Why each feature
| Feature | Why it matters |
|---|---|
| recency_days | Directly precedes churn; strongest univariate predictor |
| frequency, active_months_last_year | Habitual vs one-time buyers |
| monetary_total, avg_order_value | Core input for churn weighting and CLV |
| avg_basket_items / lines | Top-up vs bulk shopper |
| order_trend | Acceleration or deceleration hidden by averages |
| gap_mean, gap_std, overdue_ratio | Irregular customers; how late the next order is vs normal |
| q4_share, month_concentration, orders_same_season_last_year | Seasonal / gift shoppers |
| cat_* | Category affinity vector, feeds recommendations |
| return_rate | Leading indicator of dissatisfaction |
| markdown_share | Discount sensitivity: buys below the usual price |
| product_popularity | Frequency-encoded SKU mainstreamness |
| country_* | One-hot for the five largest markets |
| engagement_score | Single 0-100 RFM score for dashboards |

In [4]:
labels = build_labels(tx, cutoff, 90, f.index)
print('churn rate', labels.churn.mean().round(3))
f.join(labels).groupby('churn')[['recency_days', 'frequency', 'overdue_ratio', 'engagement_score']].median().round(1)

churn rate 0.565


,recency_days,frequency,overdue_ratio,engagement_score
churn,,,,
0,67.0,5.0,1.0,67.1
1,286.0,2.0,2.0,36.6
